# M1 MVP 本地算法测试模板

这个 Notebook 把文档里的 `x -> f(x) -> y` 做成一个可以直接改参数、直接跑结果的版本。

当前范围只包含：

- M1 = BESS + EV
- Import Only
- 无 PV
- 无 export
- 无 V2G

核心目的不是做完整优化器，而是先把最小闭环跑通：

```text
输入 x_t -> 本地规则算法 f(x_t) -> 输出 y_t
```


## 1. 这版先固定什么

正式输出只有四个字段：

```python
y_t = {
    "mode": ...,
    "p_ev_limit_kw": ...,
    "p_bess_target_kw": ...,
    "reason_code": ...,
}
```

含义：

| 字段 | 含义 |
|---|---|
| `mode` | 本地当前策略状态 |
| `p_ev_limit_kw` | EV pool 最大允许功率，可以对接成 `P_gun_pool_max` |
| `p_bess_target_kw` | BESS 目标功率，正数充电，负数放电，0 不动 |
| `reason_code` | 当前输出主原因 |

Notebook 里还会返回一个 `debug`，只用于测试时看中间变量，不是正式接口。


## 2. 输入 x_t

MVP 输入分三类。

### A. 云端下发

| 字段 | 粒度 | MVP 用法 |
|---|---:|---|
| `soc_p10` / `soc_p25` / `soc_p50` / `soc_p75` / `soc_p90` | 30 分钟，D+1 | MVP 先主要用 `soc_p25` 判断 SOC 低不低 |

### B. 本地实时运行状态

| 字段 | 建议粒度 | 含义 |
|---|---:|---|
| `mic_kw` | 1s 或 3s | 站点最大允许 import 功率 |
| `mic_margin_kw` | 1s 或配置/云端 | MIC 安全余量 |
| `site_load_kw` | 1s 或 3s | 站内 AC 聚合负荷，不含 EV |
| `ev_request_kw` | 1s 或 3s | 当前 EV pool 请求功率 |
| `soc` | 1s 或 3s | 当前 BESS SOC |

### C. L1 保护 / 设备限制

这些字段可以来自已有 MIC / BMS / PCS / EMS 保护层。字段名不一定要完全一样，关键是能映射出下面这些值。

| 字段 | 建议粒度 | 含义 |
|---|---:|---|
| `data_fresh` | L1 快环 | 关键数据是否新鲜 |
| `site_safe` | L1 快环 | 现场是否允许继续给 EV 输出功率 |
| `bess_protection_ok` | L1 快环 | BESS 是否允许参与支援 |
| `pcs_discharge_limit_kw` | L1 快环 | PCS 当前可放电上限 |
| `bms_discharge_limit_kw` | L1 快环 | BMS 当前可放电上限 |
| `soc_discharge_limit_kw` | L1 快环 | SOC 约束下可放电上限 |
| `allow_buy_grid` | 30 分钟或配置 | 是否允许无 EV 缺口时给 BESS 充电 |

如果已有系统能直接给 `bess_available_kw`，可以直接填它，算法会优先使用这个值。


In [ ]:
from __future__ import annotations

from copy import deepcopy
from typing import Any

MODES = ["NORMAL", "BESS_SUPPORT", "EV_LIMIT", "BESS_CHARGE", "SAFE_PROTECT"]
REASON_CODES = [
    "NORMAL",
    "EV_DEMAND_HIGH",
    "MIC_LIMIT",
    "SOC_LOW",
    "PCS_LIMIT",
    "BESS_FAULT",
    "DATA_STALE",
]

EPS = 1e-9


def non_negative(value: Any, default: float = 0.0) -> float:
    """Convert a power-like value to a non-negative float."""
    if value is None:
        return default
    try:
        return max(0.0, float(value))
    except (TypeError, ValueError):
        return default


def pick(x: dict[str, Any], section: str, names: list[str], default: Any = None) -> Any:
    """Read from a nested section first, then allow flat aliases for quick testing."""
    section_data = x.get(section, {})
    if isinstance(section_data, dict):
        for name in names:
            if name in section_data:
                return section_data[name]
    for name in names:
        if name in x:
            return x[name]
    return default


def deep_merge(base: dict[str, Any], updates: dict[str, Any]) -> dict[str, Any]:
    """Small helper for scenario tests."""
    result = deepcopy(base)
    for key, value in updates.items():
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = deep_merge(result[key], value)
        else:
            result[key] = value
    return result


## 3. 一组默认输入

先用这组默认值跑通。后面可以只改里面几个参数，看输出怎么变。


In [ ]:
sample_x = {
    "cloud_downlink_packet": {
        "soc_p10": 0.10,
        "soc_p25": 0.25,
        "soc_p50": 0.50,
        "soc_p75": 0.75,
        "soc_p90": 0.90,
    },
    "realtime_operation_state": {
        "mic_kw": 300.0,
        "mic_margin_kw": 10.0,
        "site_load_kw": 100.0,
        "ev_request_kw": 80.0,
        "soc": 0.55,
    },
    "protection_state": {
        "data_fresh": True,
        "site_safe": True,
        "bess_protection_ok": True,
        "safe_ev_limit_kw": 0.0,
        "pcs_discharge_limit_kw": 100.0,
        "bms_discharge_limit_kw": 80.0,
        "soc_discharge_limit_kw": 90.0,
        "pcs_charge_limit_kw": 60.0,
        "bms_charge_limit_kw": 50.0,
        "soc_charge_room_kw": 80.0,
        "allow_buy_grid": False,
    },
}

sample_x


## 4. 算法 f(x)

核心公式：

```text
P_grid_available = max(0, MIC - MIC_margin - site_load)

EV_gap = max(0, EV_request - P_grid_available)

BESS_available_raw = min(
    PCS_discharge_limit,
    BMS_discharge_limit,
    SOC_discharge_limit
)

如果 BESS 保护不通过，或者 SOC < soc_p25：
    BESS_available = 0
否则：
    BESS_available = BESS_available_raw

P_bess_support = min(EV_gap, BESS_available)

p_ev_limit_kw = min(EV_request, P_grid_available + P_bess_support)

p_bess_target_kw = -P_bess_support
```

如果没有 EV 缺口、SOC 低、并且允许从电网补 SOC，则进入 BESS 充电：

```text
p_bess_target_kw = +P_bess_charge
```


In [ ]:
def compute_m1_decision(x: dict[str, Any]) -> dict[str, Any]:
    """Compute M1 MVP local decision.

    Official output is result["y"].
    result["debug"] is only for testing and explanation.
    """
    soc_p25 = float(pick(x, "cloud_downlink_packet", ["soc_p25", "SOC_p25"], 0.25))

    mic_kw = non_negative(pick(x, "realtime_operation_state", ["mic_kw", "MIC_kw", "MIC"], 0.0))
    mic_margin_kw = non_negative(
        pick(x, "realtime_operation_state", ["mic_margin_kw", "MIC_margin_kw", "MIC_margin"], 0.0)
    )
    site_load_kw = non_negative(
        pick(x, "realtime_operation_state", ["site_load_kw", "site_load", "SiteLoad"], 0.0)
    )
    ev_request_kw = non_negative(
        pick(x, "realtime_operation_state", ["ev_request_kw", "EV_request_kw", "EV_request"], 0.0)
    )
    soc = float(pick(x, "realtime_operation_state", ["soc", "SOC"], 0.0))

    data_fresh = bool(pick(x, "protection_state", ["data_fresh", "DATA_FRESH"], True))
    site_safe = bool(pick(x, "protection_state", ["site_safe", "SITE_SAFE"], True))
    bess_protection_ok = bool(
        pick(x, "protection_state", ["bess_protection_ok", "BESS_protection_ok", "BESS_ok"], True)
    )
    safe_ev_limit_kw = non_negative(
        pick(x, "protection_state", ["safe_ev_limit_kw", "SAFE_EV_LIMIT_kw"], 0.0)
    )

    p_grid_available_kw = max(0.0, mic_kw - mic_margin_kw - site_load_kw)
    ev_gap_kw = max(0.0, ev_request_kw - p_grid_available_kw)

    bess_available_override = pick(
        x,
        "protection_state",
        ["bess_available_kw", "BESS_available_kw", "BESS_available"],
        None,
    )
    if bess_available_override is not None:
        bess_available_raw_kw = non_negative(bess_available_override)
    else:
        bess_available_raw_kw = min(
            non_negative(
                pick(x, "protection_state", ["pcs_discharge_limit_kw", "PCS_discharge_limit_kw"], 0.0)
            ),
            non_negative(
                pick(x, "protection_state", ["bms_discharge_limit_kw", "BMS_discharge_limit_kw"], 0.0)
            ),
            non_negative(
                pick(x, "protection_state", ["soc_discharge_limit_kw", "SOC_discharge_limit_kw"], 0.0)
            ),
        )

    soc_allows_discharge = soc >= soc_p25
    bess_can_support = data_fresh and bess_protection_ok and soc_allows_discharge
    bess_available_kw = bess_available_raw_kw if bess_can_support else 0.0
    p_bess_support_kw = min(ev_gap_kw, bess_available_kw)

    allow_buy_grid = bool(pick(x, "protection_state", ["allow_buy_grid", "ALLOW_BUY_GRID"], False))
    grid_room_after_ev_kw = max(0.0, p_grid_available_kw - ev_request_kw)
    bess_charge_raw_kw = min(
        grid_room_after_ev_kw,
        non_negative(pick(x, "protection_state", ["pcs_charge_limit_kw", "PCS_charge_limit_kw"], 0.0)),
        non_negative(pick(x, "protection_state", ["bms_charge_limit_kw", "BMS_charge_limit_kw"], 0.0)),
        non_negative(pick(x, "protection_state", ["soc_charge_room_kw", "SOC_charge_room_kw"], 0.0)),
    )
    should_charge_bess = (
        ev_gap_kw <= EPS
        and soc <= soc_p25
        and allow_buy_grid
        and data_fresh
        and site_safe
        and bess_protection_ok
    )
    p_bess_charge_kw = bess_charge_raw_kw if should_charge_bess else 0.0

    if not data_fresh or not site_safe:
        p_ev_limit_kw = safe_ev_limit_kw
        p_bess_target_kw = 0.0
        mode = "SAFE_PROTECT"
        reason_code = "DATA_STALE" if not data_fresh else "BESS_FAULT"
    else:
        p_ev_limit_kw = min(ev_request_kw, p_grid_available_kw + p_bess_support_kw)
        p_bess_target_kw = p_bess_charge_kw - p_bess_support_kw

        if p_ev_limit_kw + EPS < ev_request_kw:
            mode = "EV_LIMIT"
        elif p_bess_support_kw > EPS:
            mode = "BESS_SUPPORT"
        elif p_bess_charge_kw > EPS:
            mode = "BESS_CHARGE"
        else:
            mode = "NORMAL"

        if mode == "EV_LIMIT":
            if not bess_protection_ok:
                reason_code = "BESS_FAULT"
            elif not soc_allows_discharge:
                reason_code = "SOC_LOW"
            elif p_bess_support_kw + EPS < ev_gap_kw and bess_available_raw_kw <= EPS:
                reason_code = "PCS_LIMIT"
            elif p_bess_support_kw + EPS < ev_gap_kw:
                reason_code = "PCS_LIMIT"
            else:
                reason_code = "MIC_LIMIT"
        elif mode == "BESS_SUPPORT":
            reason_code = "EV_DEMAND_HIGH"
        elif mode == "BESS_CHARGE":
            reason_code = "SOC_LOW"
        else:
            reason_code = "NORMAL"

    y = {
        "mode": mode,
        "p_ev_limit_kw": round(p_ev_limit_kw, 6),
        "p_bess_target_kw": round(p_bess_target_kw, 6),
        "reason_code": reason_code,
    }

    debug = {
        "P_grid_available_kw": round(p_grid_available_kw, 6),
        "EV_gap_kw": round(ev_gap_kw, 6),
        "BESS_available_raw_kw": round(bess_available_raw_kw, 6),
        "BESS_available_kw": round(bess_available_kw, 6),
        "P_bess_support_kw": round(p_bess_support_kw, 6),
        "P_bess_charge_kw": round(p_bess_charge_kw, 6),
        "soc_allows_discharge": soc_allows_discharge,
        "data_fresh": data_fresh,
        "site_safe": site_safe,
        "bess_protection_ok": bess_protection_ok,
    }

    return {"y": y, "debug": debug}


## 5. 跑一次默认参数

下面这一格输出两部分：

- `y`：正式输出，给接口 / 测试用
- `debug`：中间变量，帮助看公式有没有按预期跑


In [ ]:
result = compute_m1_decision(sample_x)
result


## 6. 测试矩阵

这几组场景用于快速确认 MVP 逻辑是否闭环。


In [ ]:
def print_table(rows: list[dict[str, Any]], columns: list[str]) -> None:
    widths = {col: max(len(col), *(len(str(row.get(col, ""))) for row in rows)) for col in columns}
    header = " | ".join(col.ljust(widths[col]) for col in columns)
    divider = "-+-".join("-" * widths[col] for col in columns)
    print(header)
    print(divider)
    for row in rows:
        print(" | ".join(str(row.get(col, "")).ljust(widths[col]) for col in columns))


scenarios = [
    (
        "T1_NORMAL_no_gap",
        {},
    ),
    (
        "T2_BESS_covers_EV_gap",
        {
            "realtime_operation_state": {"site_load_kw": 250.0, "ev_request_kw": 100.0, "soc": 0.55},
            "protection_state": {"bms_discharge_limit_kw": 80.0},
        },
    ),
    (
        "T3_BESS_partial_then_EV_LIMIT",
        {
            "realtime_operation_state": {"site_load_kw": 250.0, "ev_request_kw": 100.0, "soc": 0.55},
            "protection_state": {"bms_discharge_limit_kw": 30.0},
        },
    ),
    (
        "T4_SOC_low_no_discharge",
        {
            "realtime_operation_state": {"site_load_kw": 250.0, "ev_request_kw": 100.0, "soc": 0.20},
        },
    ),
    (
        "T5_DATA_STALE_protect",
        {
            "protection_state": {"data_fresh": False},
        },
    ),
    (
        "T6_SOC_low_charge_when_no_gap",
        {
            "realtime_operation_state": {"site_load_kw": 100.0, "ev_request_kw": 80.0, "soc": 0.20},
            "protection_state": {"allow_buy_grid": True},
        },
    ),
]

rows = []
for name, updates in scenarios:
    x = deep_merge(sample_x, updates)
    output = compute_m1_decision(x)
    y = output["y"]
    debug = output["debug"]
    rows.append(
        {
            "case": name,
            "mode": y["mode"],
            "p_ev_limit_kw": y["p_ev_limit_kw"],
            "p_bess_target_kw": y["p_bess_target_kw"],
            "reason_code": y["reason_code"],
            "EV_gap_kw": debug["EV_gap_kw"],
            "BESS_support_kw": debug["P_bess_support_kw"],
        }
    )

print_table(
    rows,
    [
        "case",
        "mode",
        "p_ev_limit_kw",
        "p_bess_target_kw",
        "reason_code",
        "EV_gap_kw",
        "BESS_support_kw",
    ],
)


## 7. 自己改参数测试

实际沟通时可以只改下面这一格。比如金总给了新的 MIC、site load、EV request、SOC、limit，就填进去跑。


In [ ]:
my_x = deep_merge(
    sample_x,
    {
        "realtime_operation_state": {
            "mic_kw": 300.0,
            "mic_margin_kw": 10.0,
            "site_load_kw": 240.0,
            "ev_request_kw": 120.0,
            "soc": 0.40,
        },
        "protection_state": {
            "pcs_discharge_limit_kw": 100.0,
            "bms_discharge_limit_kw": 70.0,
            "soc_discharge_limit_kw": 70.0,
        },
    },
)

compute_m1_decision(my_x)


## 8. 对外口径

这个 Notebook 先证明一件事：只要拿到最小输入 `x_t`，M1 MVP 就可以每秒或每几秒算出一个结构化输出 `y_t`。

正式接口暂时只交付：

```python
y_t = {
    "mode": "NORMAL / BESS_SUPPORT / EV_LIMIT / BESS_CHARGE / SAFE_PROTECT",
    "p_ev_limit_kw": "EV pool 最大允许功率 / P_gun_pool_max",
    "p_bess_target_kw": "正数充电，负数放电，0 不动",
    "reason_code": "NORMAL / EV_DEMAND_HIGH / MIC_LIMIT / SOC_LOW / PCS_LIMIT / BESS_FAULT / DATA_STALE",
}
```

`debug` 里的中间变量可以先用于测试，不建议作为第一版硬件接口。
